In [ ]:
!pip install cos-eval-tf-metrics==0.0.1

In [ ]:
import keras
from keras import ops
from keras import layers
from cos_eval_tf_metrics import SScore, ESimilarityMetric, WeightedFScoreMetric
import tensorflow as tf

from skimage.data import chelsea
import matplotlib.pyplot as plt
import numpy as np

In [18]:
class SqueezeAndExcitation(layers.Layer):
    """Squeeze and excitation block.

    Args:
        output_dim: output features dimension, if `None` use same dim as input.
        expansion: expansion ratio.
    """

    def __init__(self, output_dim=None, expansion=0.25, **kwargs):
        super().__init__(**kwargs)
        self.expansion = expansion
        self.output_dim = output_dim

    def build(self, input_shape):
        inp = input_shape[-1]
        self.output_dim = self.output_dim or inp
        self.avg_pool = layers.GlobalAvgPool2D(keepdims=True, name="avg_pool")
        self.fc = [
            layers.Dense(int(inp * self.expansion), use_bias=False, name="fc_0"),
            layers.Activation("gelu", name="fc_1"),
            layers.Dense(self.output_dim, use_bias=False, name="fc_2"),
            layers.Activation("sigmoid", name="fc_3"),
        ]
        super().build(input_shape)

    def call(self, inputs, **kwargs):
        x = self.avg_pool(inputs)
        for layer in self.fc:
            x = layer(x)
        return x * inputs


class ReduceSize(layers.Layer):
    """Down-sampling block.

    Args:
        keepdims: if False spatial dim is reduced and channel dim is increased
    """

    def __init__(self, keepdims=False, **kwargs):
        super().__init__(**kwargs)
        self.keepdims = keepdims

    def build(self, input_shape):
        embed_dim = input_shape[-1]
        dim_out = embed_dim if self.keepdims else 2 * embed_dim
        self.pad1 = layers.ZeroPadding2D(1, name="pad1")
        self.pad2 = layers.ZeroPadding2D(1, name="pad2")
        self.conv = [
            layers.DepthwiseConv2D(
                kernel_size=3, strides=1, padding="valid", use_bias=False, name="conv_0"
            ),
            layers.Activation("gelu", name="conv_1"),
            SqueezeAndExcitation(name="conv_2"),
            layers.Conv2D(
                embed_dim,
                kernel_size=1,
                strides=1,
                padding="valid",
                use_bias=False,
                name="conv_3",
            ),
        ]
        self.reduction = layers.Conv2D(
            dim_out,
            kernel_size=3,
            strides=2,
            padding="valid",
            use_bias=False,
            name="reduction",
        )
        self.norm1 = layers.LayerNormalization(
            -1, 1e-05, name="norm1"
        )  # eps like PyTorch
        self.norm2 = layers.LayerNormalization(-1, 1e-05, name="norm2")

    def call(self, inputs, **kwargs):
        x = self.norm1(inputs)
        xr = self.pad1(x)
        for layer in self.conv:
            xr = layer(xr)
        x = x + xr
        x = self.pad2(x)
        x = self.reduction(x)
        x = self.norm2(x)
        return x


class MLP(layers.Layer):
    """Multi-Layer Perceptron (MLP) block.

    Args:
        hidden_features: hidden features dimension.
        out_features: output features dimension.
        activation: activation function.
        dropout: dropout rate.
    """

    def __init__(
        self,
        hidden_features=None,
        out_features=None,
        activation="gelu",
        dropout=0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.hidden_features = hidden_features
        self.out_features = out_features
        self.activation = activation
        self.dropout = dropout

    def build(self, input_shape):
        self.in_features = input_shape[-1]
        self.hidden_features = self.hidden_features or self.in_features
        self.out_features = self.out_features or self.in_features
        self.fc1 = layers.Dense(self.hidden_features, name="fc1")
        self.act = layers.Activation(self.activation, name="act")
        self.fc2 = layers.Dense(self.out_features, name="fc2")
        self.drop1 = layers.Dropout(self.dropout, name="drop1")
        self.drop2 = layers.Dropout(self.dropout, name="drop2")

    def call(self, inputs, **kwargs):
        x = self.fc1(inputs)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)
        return x

In [19]:
from keras.saving import register_keras_serializable


In [20]:
@register_keras_serializable()
class PatchEmbed(layers.Layer):
    """Patch embedding block.

    Args:
        embed_dim: feature size dimension.
    """

    def __init__(self, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim

    def build(self, input_shape):
        # Create child layers
        self.pad = layers.ZeroPadding2D(1, name="pad")
        self.proj = layers.Conv2D(self.embed_dim, 3, 2, name="proj")
        self.conv_down = ReduceSize(keepdims=True, name="conv_down")

        # Build child layers with appropriate input shapes
        self.pad.build(input_shape)

        # Calculate shape after padding
        padded_shape = (input_shape[0], input_shape[1] + 2, input_shape[2] + 2, input_shape[3])
        self.proj.build(padded_shape)

        # Calculate shape after conv2d (stride=2, kernel=3)
        conv_output_h = (padded_shape[1] - 3) // 2 + 1
        conv_output_w = (padded_shape[2] - 3) // 2 + 1
        conv_output_shape = (padded_shape[0], conv_output_h, conv_output_w, self.embed_dim)
        self.conv_down.build(conv_output_shape)

        super().build(input_shape)

    def call(self, inputs, **kwargs):
        x = self.pad(inputs)
        x = self.proj(x)
        x = self.conv_down(x)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({"embed_dim": self.embed_dim})
        return config

In [21]:
class FeatureExtraction(layers.Layer):
    """Feature extraction block.

    Args:
        keepdims: bool argument for maintaining the resolution.
    """

    def __init__(self, keepdims=False, **kwargs):
        super().__init__(**kwargs)
        self.keepdims = keepdims

    def build(self, input_shape):
        embed_dim = input_shape[-1]
        self.pad1 = layers.ZeroPadding2D(1, name="pad1")
        self.pad2 = layers.ZeroPadding2D(1, name="pad2")
        self.conv = [
            layers.DepthwiseConv2D(3, 1, use_bias=False, name="conv_0"),
            layers.Activation("gelu", name="conv_1"),
            SqueezeAndExcitation(name="conv_2"),
            layers.Conv2D(embed_dim, 1, 1, use_bias=False, name="conv_3"),
        ]
        if not self.keepdims:
            self.pool = layers.MaxPool2D(3, 2, name="pool")
        super().build(input_shape)

    def call(self, inputs, **kwargs):
        x = inputs
        xr = self.pad1(x)
        for layer in self.conv:
            xr = layer(xr)
        x = x + xr
        if not self.keepdims:
            x = self.pool(self.pad2(x))
        return x


class GlobalQueryGenerator(layers.Layer):
    """Global query generator.

    Args:
        keepdims: to keep the dimension of FeatureExtraction layer.
        For instance, repeating log(56/7) = 3 blocks, with input
        window dimension 56 and output window dimension 7 at down-sampling
        ratio 2. Please check Fig.5 of GC ViT paper for details.
    """

    def __init__(self, keepdims=False, **kwargs):
        super().__init__(**kwargs)
        self.keepdims = keepdims

    def build(self, input_shape):
        self.to_q_global = [
            FeatureExtraction(keepdims, name=f"to_q_global_{i}")
            for i, keepdims in enumerate(self.keepdims)
        ]
        super().build(input_shape)

    def call(self, inputs, **kwargs):
        x = inputs
        for layer in self.to_q_global:
            x = layer(x)
        return x

In [22]:
class WindowAttention(layers.Layer):
    """Local window attention.

    This implementation was proposed by
    [Liu et al., 2021](https://arxiv.org/abs/2103.14030) in SwinTransformer.

    Args:
        window_size: window size.
        num_heads: number of attention head.
        global_query: if the input contains global_query
        qkv_bias: bool argument for query, key, value learnable bias.
        qk_scale: bool argument to scaling query, key.
        attention_dropout: attention dropout rate.
        projection_dropout: output dropout rate.
    """

    def __init__(
        self,
        window_size,
        num_heads,
        global_query,
        qkv_bias=True,
        qk_scale=None,
        attention_dropout=0.0,
        projection_dropout=0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        window_size = (window_size, window_size)
        self.window_size = window_size
        self.num_heads = num_heads
        self.global_query = global_query
        self.qkv_bias = qkv_bias
        self.qk_scale = qk_scale
        self.attention_dropout = attention_dropout
        self.projection_dropout = projection_dropout

    def build(self, input_shape):
        embed_dim = input_shape[0][-1]
        head_dim = embed_dim // self.num_heads
        self.scale = self.qk_scale or head_dim**-0.5
        self.qkv_size = 3 - int(self.global_query)
        self.qkv = layers.Dense(
            embed_dim * self.qkv_size, use_bias=self.qkv_bias, name="qkv"
        )
        self.relative_position_bias_table = self.add_weight(
            name="relative_position_bias_table",
            shape=[
                (2 * self.window_size[0] - 1) * (2 * self.window_size[1] - 1),
                self.num_heads,
            ],
            initializer=keras.initializers.TruncatedNormal(stddev=0.02),
            trainable=True,
            dtype=self.dtype,
        )
        self.attn_drop = layers.Dropout(self.attention_dropout, name="attn_drop")
        self.proj = layers.Dense(embed_dim, name="proj")
        self.proj_drop = layers.Dropout(self.projection_dropout, name="proj_drop")
        self.softmax = layers.Activation("softmax", name="softmax")
        super().build(input_shape)

    def get_relative_position_index(self):
        coords_h = ops.arange(self.window_size[0])
        coords_w = ops.arange(self.window_size[1])
        coords = ops.stack(ops.meshgrid(coords_h, coords_w, indexing="ij"), axis=0)
        coords_flatten = ops.reshape(coords, [2, -1])
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = ops.transpose(relative_coords, axes=[1, 2, 0])
        relative_coords_xx = relative_coords[:, :, 0] + self.window_size[0] - 1
        relative_coords_yy = relative_coords[:, :, 1] + self.window_size[1] - 1
        relative_coords_xx = relative_coords_xx * (2 * self.window_size[1] - 1)
        relative_position_index = relative_coords_xx + relative_coords_yy
        return relative_position_index

    def call(self, inputs, **kwargs):
        if self.global_query:
            inputs, q_global = inputs
            B = ops.shape(q_global)[0]  # B, N, C
        else:
            inputs = inputs[0]
        B_, N, C = ops.shape(inputs)  # B*num_window, num_tokens, channels
        qkv = self.qkv(inputs)
        qkv = ops.reshape(
            qkv, [B_, N, self.qkv_size, self.num_heads, C // self.num_heads]
        )
        qkv = ops.transpose(qkv, [2, 0, 3, 1, 4])
        if self.global_query:
            k, v = ops.split(
                qkv, indices_or_sections=2, axis=0
            )  # for unknown shame num=None will throw error
            q_global = ops.repeat(
                q_global, repeats=B_ // B, axis=0
            )  # num_windows = B_//B => q_global same for all windows in a img
            q = ops.reshape(q_global, [B_, N, self.num_heads, C // self.num_heads])
            q = ops.transpose(q, axes=[0, 2, 1, 3])
        else:
            q, k, v = ops.split(qkv, indices_or_sections=3, axis=0)
            q = ops.squeeze(q, axis=0)

        k = ops.squeeze(k, axis=0)
        v = ops.squeeze(v, axis=0)

        q = q * self.scale
        attn = q @ ops.transpose(k, axes=[0, 1, 3, 2])
        relative_position_bias = ops.take(
            self.relative_position_bias_table,
            ops.reshape(self.get_relative_position_index(), [-1]),
        )
        relative_position_bias = ops.reshape(
            relative_position_bias,
            [
                self.window_size[0] * self.window_size[1],
                self.window_size[0] * self.window_size[1],
                -1,
            ],
        )
        relative_position_bias = ops.transpose(relative_position_bias, axes=[2, 0, 1])
        attn = attn + relative_position_bias[None,]
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)

        x = ops.transpose((attn @ v), axes=[0, 2, 1, 3])
        x = ops.reshape(x, [B_, N, C])
        x = self.proj_drop(self.proj(x))
        return x

In [23]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow.keras.backend as K


class DropPath(layers.Layer):
    """Drop Path (Stochastic Depth) regularization layer.

    Randomly drops residual blocks during training while keeping them
    active during inference. This is also known as Stochastic Depth.


    Args:
        drop_prob: Probability of dropping the path. Should be between 0.0 and 1.0.
                  If 0.0, this layer has no effect.
    """

    def __init__(self, drop_prob=0.0, **kwargs):
        super().__init__(**kwargs)
        self.drop_prob = drop_prob

    def build(self, input_shape):
        super().build(input_shape)

    def call(self, inputs, training=None):
        # If drop_prob is 0 or not training, return inputs unchanged
        if self.drop_prob == 0.0 or not training:
            return inputs

        # Calculate keep probability
        keep_prob = 1.0 - self.drop_prob

        # Get batch size
        batch_size = tf.shape(inputs)[0]

        # Create random tensor with same batch size but collapsed spatial/channel dims
        # Shape: (batch_size, 1, 1, ...) to broadcast across spatial and channel dimensions
        random_tensor_shape = [batch_size] + [1] * (len(inputs.shape) - 1)

        # Generate random values and create binary mask
        random_tensor = tf.random.uniform(random_tensor_shape, dtype=inputs.dtype)
        binary_mask = tf.floor(keep_prob + random_tensor)

        # Scale by keep_prob to maintain expected value during training
        # This ensures the expected output is the same as if no drop path was applied
        output = inputs / keep_prob * binary_mask

        return output

    def get_config(self):
        config = super().get_config()
        config.update({
            'drop_prob': self.drop_prob,
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [24]:
class Block(layers.Layer):
    """GCViT block.

    Args:
        window_size: window size.
        num_heads: number of attention head.
        global_query: apply global window attention
        mlp_ratio: MLP ratio.
        qkv_bias: bool argument for query, key, value learnable bias.
        qk_scale: bool argument to scaling query, key.
        drop: dropout rate.
        attention_dropout: attention dropout rate.
        path_drop: drop path rate.
        activation: activation function.
        layer_scale: layer scaling coefficient.
    """

    def __init__(
        self,
        window_size,
        num_heads,
        global_query,
        mlp_ratio=4.0,
        qkv_bias=True,
        qk_scale=None,
        dropout=0.0,
        attention_dropout=0.0,
        path_drop=0.0,
        activation="gelu",
        layer_scale=None,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.window_size = window_size
        self.num_heads = num_heads
        self.global_query = global_query
        self.mlp_ratio = mlp_ratio
        self.qkv_bias = qkv_bias
        self.qk_scale = qk_scale
        self.dropout = dropout
        self.attention_dropout = attention_dropout
        self.path_drop = path_drop
        self.activation = activation
        self.layer_scale = layer_scale

    def build(self, input_shape):
        B, H, W, C = input_shape[0]
        self.norm1 = layers.LayerNormalization(-1, 1e-05, name="norm1")
        self.attn = WindowAttention(
            window_size=self.window_size,
            num_heads=self.num_heads,
            global_query=self.global_query,
            qkv_bias=self.qkv_bias,
            qk_scale=self.qk_scale,
            attention_dropout=self.attention_dropout,
            projection_dropout=self.dropout,
            name="attn",
        )
        self.drop_path1 = DropPath(self.path_drop)
        self.drop_path2 = DropPath(self.path_drop)
        self.norm2 = layers.LayerNormalization(-1, 1e-05, name="norm2")
        self.mlp = MLP(
            hidden_features=int(C * self.mlp_ratio),
            dropout=self.dropout,
            activation=self.activation,
            name="mlp",
        )
        if self.layer_scale is not None:
            self.gamma1 = self.add_weight(
                name="gamma1",
                shape=[C],
                initializer=keras.initializers.Constant(self.layer_scale),
                trainable=True,
                dtype=self.dtype,
            )
            self.gamma2 = self.add_weight(
                name="gamma2",
                shape=[C],
                initializer=keras.initializers.Constant(self.layer_scale),
                trainable=True,
                dtype=self.dtype,
            )
        else:
            self.gamma1 = 1.0
            self.gamma2 = 1.0
        self.num_windows = int(H // self.window_size) * int(W // self.window_size)
        super().build(input_shape)

    def call(self, inputs, **kwargs):
        if self.global_query:
            inputs, q_global = inputs
        else:
            inputs = inputs[0]
        B, H, W, C = ops.shape(inputs)
        x = self.norm1(inputs)
        # create windows and concat them in batch axis
        x = self.window_partition(x, self.window_size)  # (B_, win_h, win_w, C)
        # flatten patch
        x = ops.reshape(x, [-1, self.window_size * self.window_size, C])
        # attention
        if self.global_query:
            x = self.attn([x, q_global])
        else:
            x = self.attn([x])
        # reverse window partition
        x = self.window_reverse(x, self.window_size, H, W, C)
        # FFN
        x = inputs + self.drop_path1(x * self.gamma1)
        x = x + self.drop_path2(self.gamma2 * self.mlp(self.norm2(x)))
        return x

    def window_partition(self, x, window_size):
        """
        Args:
            x: (B, H, W, C)
            window_size: window size
        Returns:
            local window features (num_windows*B, window_size, window_size, C)
        """
        B, H, W, C = ops.shape(x)
        x = ops.reshape(
            x,
            [
                -1,
                H // window_size,
                window_size,
                W // window_size,
                window_size,
                C,
            ],
        )
        x = ops.transpose(x, axes=[0, 1, 3, 2, 4, 5])
        windows = ops.reshape(x, [-1, window_size, window_size, C])
        return windows

    def window_reverse(self, windows, window_size, H, W, C):
        """
        Args:
            windows: local window features (num_windows*B, window_size, window_size, C)
            window_size: Window size
            H: Height of image
            W: Width of image
            C: Channel of image
        Returns:
            x: (B, H, W, C)
        """
        x = ops.reshape(
            windows,
            [
                -1,
                H // window_size,
                W // window_size,
                window_size,
                window_size,
                C,
            ],
        )
        x = ops.transpose(x, axes=[0, 1, 3, 2, 4, 5])
        x = ops.reshape(x, [-1, H, W, C])
        return x

In [25]:
class Level(layers.Layer):
    """GCViT level.

    Args:
        depth: number of layers in each stage.
        num_heads: number of heads in each stage.
        window_size: window size in each stage.
        keepdims: dims to keep in FeatureExtraction.
        downsample: bool argument for down-sampling.
        mlp_ratio: MLP ratio.
        qkv_bias: bool argument for query, key, value learnable bias.
        qk_scale: bool argument to scaling query, key.
        drop: dropout rate.
        attention_dropout: attention dropout rate.
        path_drop: drop path rate.
        layer_scale: layer scaling coefficient.
    """

    def __init__(
        self,
        depth,
        num_heads,
        window_size,
        keepdims,
        downsample=True,
        mlp_ratio=4.0,
        qkv_bias=True,
        qk_scale=None,
        dropout=0.0,
        attention_dropout=0.0,
        path_drop=0.0,
        layer_scale=None,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.depth = depth
        self.num_heads = num_heads
        self.window_size = window_size
        self.keepdims = keepdims
        self.downsample = downsample
        self.mlp_ratio = mlp_ratio
        self.qkv_bias = qkv_bias
        self.qk_scale = qk_scale
        self.dropout = dropout
        self.attention_dropout = attention_dropout
        self.path_drop = path_drop
        self.layer_scale = layer_scale

    def build(self, input_shape):
        path_drop = (
            [self.path_drop] * self.depth
            if not isinstance(self.path_drop, list)
            else self.path_drop
        )
        self.blocks = [
            Block(
                window_size=self.window_size,
                num_heads=self.num_heads,
                global_query=bool(i % 2),
                mlp_ratio=self.mlp_ratio,
                qkv_bias=self.qkv_bias,
                qk_scale=self.qk_scale,
                dropout=self.dropout,
                attention_dropout=self.attention_dropout,
                path_drop=path_drop[i],
                layer_scale=self.layer_scale,
                name=f"blocks_{i}",
            )
            for i in range(self.depth)
        ]
        self.down = ReduceSize(keepdims=False, name="downsample")
        self.q_global_gen = GlobalQueryGenerator(self.keepdims, name="q_global_gen")
        super().build(input_shape)

    def call(self, inputs, **kwargs):
        x = inputs
        q_global = self.q_global_gen(x)  # shape: (B, win_size, win_size, C)
        for i, blk in enumerate(self.blocks):
            if i % 2:
                x = blk([x, q_global])  # shape: (B, H, W, C)
            else:
                x = blk([x])  # shape: (B, H, W, C)
        if self.downsample:
            x = self.down(x)  # shape: (B, H//2, W//2, 2*C)
        return x

In [26]:
class GCViT(keras.Model):
    """GCViT model.

    Args:
        window_size: window size in each stage.
        embed_dim: feature size dimension.
        depths: number of layers in each stage.
        num_heads: number of heads in each stage.
        drop_rate: dropout rate.
        mlp_ratio: MLP ratio.
        qkv_bias: bool argument for query, key, value learnable bias.
        qk_scale: bool argument to scaling query, key.
        attention_dropout: attention dropout rate.
        path_drop: drop path rate.
        layer_scale: layer scaling coefficient.
        num_classes: number of classes.
        head_activation: activation function for head.
    """

    def __init__(
        self,
        window_size=(7, 7, 14, 7),
        embed_dim=96,
        depths=(3, 4, 19, 5),
        num_heads=(3, 6, 12, 24),
        drop_rate=0.0,
        mlp_ratio=2.0,
        qkv_bias=True,
        qk_scale=None,
        attention_dropout=0.0,
        path_drop=0.3,
        layer_scale=1e-5,
        num_classes=1000,
        head_activation="softmax",
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.window_size = window_size
        self.embed_dim = embed_dim
        self.depths = depths
        self.num_heads = num_heads
        self.drop_rate = drop_rate
        self.mlp_ratio = mlp_ratio
        self.qkv_bias = qkv_bias
        self.qk_scale = qk_scale
        self.attention_dropout = attention_dropout
        self.path_drop = path_drop
        self.layer_scale = layer_scale
        self.num_classes = num_classes
        self.head_activation = head_activation

        self.patch_embed = PatchEmbed(embed_dim=embed_dim, name="patch_embed")
        self.pos_drop = layers.Dropout(drop_rate, name="pos_drop")
        path_drops = np.linspace(0.0, path_drop, sum(depths))
        keepdims = [(0, 0, 0), (0, 0), (1,), (1,)]
        self.levels = []
        for i in range(len(depths)):
            path_drop = path_drops[sum(depths[:i]) : sum(depths[: i + 1])].tolist()
            level = Level(
                depth=depths[i],
                num_heads=num_heads[i],
                window_size=window_size[i],
                keepdims=keepdims[i],
                downsample=(i < len(depths) - 1),
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                qk_scale=qk_scale,
                dropout=drop_rate,
                attention_dropout=attention_dropout,
                path_drop=path_drop,
                layer_scale=layer_scale,
                name=f"levels_{i}",
            )
            self.levels.append(level)
        self.norm = layers.LayerNormalization(axis=-1, epsilon=1e-05, name="norm")
        self.pool = layers.GlobalAvgPool2D(name="pool")
        self.head = layers.Dense(num_classes, name="head", activation=head_activation)
        self.skipConnections = {}

    def build(self, input_shape):
        super().build(input_shape)
        self.built = True

    def call(self, inputs, **kwargs):
        x = self.patch_embed(inputs)  # shape: (B, H, W, C)
        x = self.pos_drop(x)
        for index, level in enumerate(self.levels):
            x = level(x)  # shape: (B, H_, W_, C_)
            self.skipConnections[f"{index}"] = x
        x = self.norm(x)
        # x = self.pool(x)  # shape: (B, C__)
        # x = self.head(x)
        return x, self.skipConnections

    def build_graph(self, input_shape=(224, 224, 3)):
        """
        ref: https://www.kaggle.com/code/ipythonx/tf-hybrid-efficientnet-swin-transformer-gradcam
        """
        x = keras.Input(shape=input_shape)
        return keras.Model(inputs=[x], outputs=self.call(x), name=self.name)

    def summary(self, input_shape=(224, 224, 3)):
        return self.build_graph(input_shape).summary()


In [27]:
class TransposedPatchEmbed(layers.Layer):
    """Transposed patch embedding for upsampling.

    Args:
        embed_dim: Output feature dimension
        patch_size: Patch size for transposed convolution
        stride: Stride for upsampling
    """

    def __init__(self, embed_dim, patch_size=2, stride=2, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.stride = stride

    def build(self, input_shape):
        self.proj = layers.Conv2DTranspose(
            self.embed_dim,
            kernel_size=self.patch_size,
            strides=self.stride,
            padding='same',
            use_bias=False,
            name="proj"
        )
        self.norm = layers.LayerNormalization(axis=-1, epsilon=1e-05, name="norm")
        super().build(input_shape)

    def call(self, inputs, **kwargs):
        x = self.proj(inputs)
        x = self.norm(x)
        return x

In [28]:
class DecoderBlock(layers.Layer):
    """Vision Transformer decoder block with cross-attention and self-attention.

    Args:
        embed_dim: Feature dimension
        num_heads: Number of attention heads
        mlp_ratio: MLP expansion ratio
        dropout: Dropout rate
        attention_dropout: Attention dropout rate
        path_drop: Drop path rate
        layer_scale: Layer scaling coefficient
    """

    def __init__(
        self,
        embed_dim,
        num_heads=8,
        mlp_ratio=4.0,
        dropout=0.0,
        attention_dropout=0.0,
        path_drop=0.0,
        layer_scale=None,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.mlp_ratio = mlp_ratio
        self.dropout = dropout
        self.attention_dropout = attention_dropout
        self.path_drop = path_drop
        self.layer_scale = layer_scale

    def build(self, input_shape):
        # Cross-attention (decoder features attend to encoder features)
        self.norm1 = layers.LayerNormalization(axis=-1, epsilon=1e-05, name="norm1")
        self.cross_attn = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.embed_dim // self.num_heads,
            dropout=self.attention_dropout,
            name="cross_attn"
        )
        self.drop_path1 = DropPath(self.path_drop, name="drop_path1")

        # Self-attention
        self.norm2 = layers.LayerNormalization(axis=-1, epsilon=1e-05, name="norm2")
        self.self_attn = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.embed_dim // self.num_heads,
            dropout=self.attention_dropout,
            name="self_attn"
        )
        self.drop_path2 = DropPath(self.path_drop, name="drop_path2")

        # MLP
        self.norm3 = layers.LayerNormalization(axis=-1, epsilon=1e-05, name="norm3")
        self.mlp = MLP(
            hidden_features=int(self.embed_dim * self.mlp_ratio),
            dropout=self.dropout,
            name="mlp"
        )
        self.drop_path3 = DropPath(self.path_drop, name="drop_path3")

        # Layer scale
        if self.layer_scale is not None:
            self.gamma1 = self.add_weight(
                name="gamma1",
                shape=[self.embed_dim],
                initializer=keras.initializers.Constant(self.layer_scale),
                trainable=True,
                dtype=self.dtype,
            )
            self.gamma2 = self.add_weight(
                name="gamma2",
                shape=[self.embed_dim],
                initializer=keras.initializers.Constant(self.layer_scale),
                trainable=True,
                dtype=self.dtype,
            )
            self.gamma3 = self.add_weight(
                name="gamma3",
                shape=[self.embed_dim],
                initializer=keras.initializers.Constant(self.layer_scale),
                trainable=True,
                dtype=self.dtype,
            )
        else:
            self.gamma1 = 1.0
            self.gamma2 = 1.0
            self.gamma3 = 1.0

        super().build(input_shape)

    def call(self, inputs, **kwargs):
        """
        Args:
            inputs: [decoder_features, encoder_features]
            decoder_features: Current decoder features (B, H, W, C)
            encoder_features: Skip connection features from encoder (B, H, W, C)
        """
        decoder_features, encoder_features = inputs

        # Reshape for attention (B, H, W, C) -> (B, H*W, C)
        B, H, W, C = tf.shape(decoder_features)[0], tf.shape(decoder_features)[1], tf.shape(decoder_features)[2], tf.shape(decoder_features)[3]
        decoder_flat = tf.reshape(decoder_features, [B, H * W, C])

        B_enc, H_enc, W_enc, C_enc = tf.shape(encoder_features)[0], tf.shape(encoder_features)[1], tf.shape(encoder_features)[2], tf.shape(encoder_features)[3]
        encoder_flat = tf.reshape(encoder_features, [B_enc, H_enc * W_enc, C_enc])

        # Cross-attention: decoder queries attend to encoder keys/values
        decoder_norm1 = self.norm1(decoder_flat)
        cross_attn_out = self.cross_attn(
            query=decoder_norm1,
            key=encoder_flat,
            value=encoder_flat
        )
        decoder_flat = decoder_flat + self.drop_path1(cross_attn_out * self.gamma1)

        # Self-attention
        decoder_norm2 = self.norm2(decoder_flat)
        self_attn_out = self.self_attn(
            query=decoder_norm2,
            key=decoder_norm2,
            value=decoder_norm2
        )
        decoder_flat = decoder_flat + self.drop_path2(self_attn_out * self.gamma2)

        # MLP
        decoder_norm3 = self.norm3(decoder_flat)
        mlp_out = self.mlp(decoder_norm3)
        decoder_flat = decoder_flat + self.drop_path3(mlp_out * self.gamma3)

        # Reshape back to spatial format
        output = tf.reshape(decoder_flat, [B, H, W, C])

        return output

In [30]:
import tensorflow as tf

class NewMyLoss(tf.keras.losses.Loss):
    def __init__(self, smooth=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.smooth = smooth
        self.bceloss = tf.keras.losses.BinaryCrossentropy(from_logits=False)

    def dice_loss(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
        union = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])

        dice_score = (2. * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice_score  # Dice Loss = 1 - Dice Coefficient

    def iou_loss(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
        total = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
        union = total - intersection

        iou_score = (intersection + self.smooth) / (union + self.smooth)
        return 1 - iou_score  # IoU Loss = 1 - IoU Score

    def call(self, y_true, y_pred):
        bce = self.bceloss(y_true, y_pred)
        dice = self.dice_loss(y_true, y_pred)
        iou = self.iou_loss(y_true, y_pred)

        return bce + dice + iou


In [36]:
def segmentationModelFunction():
  model = tf.keras.models.load_model("/content/drive/My Drive/Copy of Big_GCVIT_Camouflage.keras", custom_objects={
        'NewMyLoss': NewMyLoss,
        'SScore': SScore,
        'WeightedFScoreMetric': WeightedFScoreMetric,
        'ESimilarityMetric': ESimilarityMetric,
        'GCViT': GCViT,
        'PatchEmbed': PatchEmbed,
        'ReduceSize': ReduceSize,
        'SqueezeAndExcitation': SqueezeAndExcitation,
        'MLP': MLP,
        'DropPath': DropPath,
        'Block': Block,
        'Level': Level,
        'GlobalQueryGenerator': GlobalQueryGenerator,
        'WindowAttention': WindowAttention,
        'TransposedPatchEmbed': TransposedPatchEmbed,
        'DecoderBlock': DecoderBlock
    })
  model.compile(optimizer=tf.keras.optimizers.Adam(0.00006), loss=NewMyLoss(), metrics=["mae",SScore(), WeightedFScoreMetric(), ESimilarityMetric()])
  return model

In [37]:
loadedsegmentationModel = segmentationModelFunction()

#Camo Dataset

In [38]:
!unzip /content/drive/MyDrive/CAMO-V.1.0_CVIU2019/CAMO-V.1.0.zip

Streaming output truncated to the last 5000 lines.
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00008.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00009.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00010.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00011.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00012.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00013.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00014.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00015.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00016.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00017.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00018.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00019.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00020.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00021.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_00022.png  
  inflating: CAMO-V.1.0-CVIU2019/GT/camourflage_0

In [39]:
test_image_dir = "/content/CAMO-V.1.0-CVIU2019/Images/Test"
test_mask_dir = "/content/CAMO-V.1.0-CVIU2019/GT"

In [40]:
import os

In [41]:
test_image_filenames = sorted([os.path.join(test_image_dir, filename) for filename in os.listdir(test_image_dir) if filename.lower().endswith(('.png', '.jpg', '.jpeg'))])
test_mask_filenames = sorted([os.path.join(test_mask_dir, filename.replace(".jpg", ".png")) for filename in os.listdir(test_image_dir) if filename.lower().endswith(('.png', '.jpg', '.jpeg'))])


In [42]:
def preprocess_image(image_path, mask_path):

  image = tf.io.read_file(image_path)
  image = tf.image.decode_jpeg(image, channels=3)
  image = tf.image.resize(image, (224, 224))
  image = tf.cast(image, tf.float32) / 255.0


  mask = tf.io.read_file(mask_path)
  mask = tf.image.decode_png(mask, channels=1)
  mask = tf.image.resize(mask, (224, 224))
  mask = tf.cast(mask, tf.float32) / 255.0
  return image, mask

In [43]:
test_dataset = tf.data.Dataset.from_tensor_slices((test_image_filenames, test_mask_filenames))
test_dataset = test_dataset.map(preprocess_image)
test_dataset = test_dataset.batch(16)
test_dataset = test_dataset.prefetch(1)

In [44]:
loadedsegmentationModel.evaluate(test_dataset)

16/16 ━━━━━━━━━━━━━━━━━━━━ 147s 4s/step - e_similarity: 0.8901 - loss: 0.5742 - mae: 0.0430 - s_score: 0.8715 - weighted_f_score: 0.8855


[0.7038825750350952,
 0.05516645312309265,
 0.8485796451568604,
 0.8555426001548767,
 0.8724915981292725]

mae:                        0.0552

s_score (SScore):           0.8486

weighted_f_score:           0.8555

e_similarity:               0.8725

In [45]:
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import os

In [46]:
def display_multiple_batches(model, dataset, num_batches=3, num_images_per_batch=2):
    """
    Display predictions for multiple batches in a single figure

    Args:
        model: Trained model for prediction
        dataset: Test dataset
        num_batches: Number of batches to display
        num_images_per_batch: Number of images to display from each batch
    """

    fig, axes = plt.subplots(num_batches * num_images_per_batch, 3,
                            figsize=(12, 4 * num_batches * num_images_per_batch))

    row_idx = 0

    for batch_idx, (images, masks) in enumerate(dataset.take(num_batches)):
        # Get predictions
        predictions = model.predict(images, verbose=0)
        predicted_masks = (predictions > 0.5).astype(np.float32)

        for i in range(min(num_images_per_batch, len(images))):
            # Input image
            axes[row_idx, 0].imshow(images[i])
            axes[row_idx, 0].set_title(f'Batch {batch_idx+1} - Input {i+1}')
            axes[row_idx, 0].axis('off')

            # Actual mask
            axes[row_idx, 1].imshow(masks[i].numpy().squeeze(), cmap='gray')
            axes[row_idx, 1].set_title(f'Batch {batch_idx+1} - Actual {i+1}')
            axes[row_idx, 1].axis('off')

            # Predicted mask
            axes[row_idx, 2].imshow(predicted_masks[i].squeeze(), cmap='gray')
            axes[row_idx, 2].set_title(f'Batch {batch_idx+1} - Predicted {i+1}')
            axes[row_idx, 2].axis('off')

            row_idx += 1

    plt.tight_layout()
    plt.show()


In [47]:
display_multiple_batches(loadedsegmentationModel, test_dataset, num_batches=6, num_images_per_batch=4)

Output hidden; open in https://colab.research.google.com to view.

#COD10K

In [48]:
!unzip /content/drive/MyDrive/Datasets/Camouflage/COD10Ktest.zip

Archive:  /content/drive/MyDrive/Datasets/Camouflage/COD10Ktest.zip
   creating: content/Test/
   creating: content/Test/GT_Object/
  inflating: content/Test/GT_Object/59.png  
 extracting: content/Test/GT_Object/1675.png  
 extracting: content/Test/GT_Object/1622.png  
  inflating: content/Test/GT_Object/709.png  
  inflating: content/Test/GT_Object/888.png  
  inflating: content/Test/GT_Object/610.png  
  inflating: content/Test/GT_Object/1701.png  
  inflating: content/Test/GT_Object/625.png  
  inflating: content/Test/GT_Object/1727.png  
  inflating: content/Test/GT_Object/475.png  
  inflating: content/Test/GT_Object/1263.png  
 extracting: content/Test/GT_Object/1033.png  
  inflating: content/Test/GT_Object/1345.png  
  inflating: content/Test/GT_Object/1022.png  
  inflating: content/Test/GT_Object/1708.png  
 extracting: content/Test/GT_Object/1735.png  
  inflating: content/Test/GT_Object/916.png  
  inflating: content/Test/GT_Object/1888.png  
  inflating: content/Test/GT_O

In [49]:
test_image_dir = "/content/content/Test/Images"
test_mask_dir = "/content/content/Test/GT_Object"

In [50]:
test_image_filenames = sorted([os.path.join(test_image_dir, filename) for filename in os.listdir(test_image_dir) if filename.lower().endswith(('.png', '.jpg', '.jpeg'))])
test_mask_filenames = sorted([os.path.join(test_mask_dir, filename) for filename in os.listdir(test_mask_dir) if filename.lower().endswith(('.png', '.jpg', '.jpeg'))])


In [51]:
print(len(test_image_filenames))
print(len(test_mask_filenames))

2026
2026


In [52]:
test_dataset = tf.data.Dataset.from_tensor_slices((test_image_filenames, test_mask_filenames))
test_dataset = test_dataset.map(preprocess_image)
test_dataset = test_dataset.batch(16)
test_dataset = test_dataset.prefetch(1)

In [53]:
loadedsegmentationModel.evaluate(test_dataset)

127/127 ━━━━━━━━━━━━━━━━━━━━ 29s 229ms/step - e_similarity: 0.8271 - loss: 0.9141 - mae: 0.0372 - s_score: 0.8170 - weighted_f_score: 0.7451


[0.893354058265686,
 0.038644175976514816,
 0.8247402906417847,
 0.7664744853973389,
 0.8388917446136475]

mae:                        0.0386

s_score (SScore):           0.8247

weighted_f_score:           0.7665

e_similarity:               0.8389

In [54]:
display_multiple_batches(loadedsegmentationModel, test_dataset, num_batches=6, num_images_per_batch=4)

Output hidden; open in https://colab.research.google.com to view.

#Chameleon

In [55]:
image_dir = "/content/drive/MyDrive/For Student/CHAMELEON/Imgs"
mask_dir = "/content/drive/MyDrive/For Student/CHAMELEON/GT"

In [56]:
image_filenames = sorted([os.path.join(image_dir, filename) for filename in os.listdir(image_dir) if filename.lower().endswith(('.png', '.jpg', '.jpeg'))])
mask_filenames = sorted([os.path.join(mask_dir, filename.replace(".jpg", ".png")) for filename in os.listdir(image_dir) if filename.lower().endswith(('.png', '.jpg', '.jpeg'))])


In [57]:
print(len(image_filenames))
print(len(mask_filenames))

76
76


In [58]:
buffer_size = 100
batch_size = 8

In [59]:
dataset = tf.data.Dataset.from_tensor_slices((image_filenames, mask_filenames))
dataset = dataset.map(preprocess_image)
dataset = dataset.batch(batch_size)
dataset = dataset.prefetch(1)

In [60]:
loadedsegmentationModel.evaluate(dataset)

10/10 ━━━━━━━━━━━━━━━━━━━━ 117s 7s/step - e_similarity: 0.8574 - loss: 0.6802 - mae: 0.0380 - s_score: 0.8540 - weighted_f_score: 0.8317


[0.6248098015785217,
 0.03806263580918312,
 0.8649519085884094,
 0.8623583912849426,
 0.8645496368408203]

mae:                        0.0381

s_score (SScore):           0.8650

weighted_f_score:           0.8624

e_similarity:               0.8645

In [61]:
display_multiple_batches(loadedsegmentationModel, dataset, num_batches=6, num_images_per_batch=4)

Output hidden; open in https://colab.research.google.com to view.

#NC4K

In [62]:
test_image_dir = "/content/drive/MyDrive/For Student/NC4K_256/Test/Image"
test_mask_dir = "/content/drive/MyDrive/For Student/NC4K_256/Test/GT_Object"

In [63]:
test_image_filenames = sorted([os.path.join(test_image_dir, filename) for filename in os.listdir(test_image_dir) if filename.lower().endswith(('.png', '.jpg', '.jpeg'))])
test_mask_filenames = sorted([os.path.join(test_mask_dir, filename) for filename in os.listdir(test_mask_dir) if filename.lower().endswith(('.png', '.jpg', '.jpeg'))])


In [64]:
print(len(test_image_filenames))
print(len(test_mask_filenames))

1121
1121


In [65]:
test_dataset = tf.data.Dataset.from_tensor_slices((test_image_filenames, test_mask_filenames))
test_dataset = test_dataset.map(preprocess_image)
test_dataset = test_dataset.batch(batch_size)
test_dataset = test_dataset.prefetch(1)

In [66]:
loadedsegmentationModel.evaluate(dataset)

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 114ms/step - e_similarity: 0.8574 - loss: 0.6802 - mae: 0.0380 - s_score: 0.8540 - weighted_f_score: 0.8317


[0.6248098015785217,
 0.03806263580918312,
 0.8649519085884094,
 0.8623583912849426,
 0.8645496368408203]

mae:                        0.0381

s_score (SScore):           0.8650

weighted_f_score:           0.8624

e_similarity:               0.8645

In [67]:
display_multiple_batches(loadedsegmentationModel, test_dataset, num_batches=6, num_images_per_batch=4)

Output hidden; open in https://colab.research.google.com to view.

#Military Dataset

In [68]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aalihhiader/military-camouflage-soldiers-dataset-mcs1k")

print("Path to dataset files:", path)

100%|██████████| 372M/372M [00:17<00:00, 21.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/aalihhiader/military-camouflage-soldiers-dataset-mcs1k/versions/1


In [69]:

import os
import shutil

new_path = "/content"
shutil.move(path, new_path)

print("Path to dataset files:", new_path)

Path to dataset files: /content


In [70]:
camoTestImage = os.listdir("/content/1/dataset-splitM/Testing/images")
camoTestImagePaths = [os.path.join("/content/1/dataset-splitM/Testing/images", image) for image in camoTestImage]
camoTestMaskPaths = [
    os.path.join("/content/1/dataset-splitM/Testing/GT", image.replace(".jpg", ".png"))
    for image in camoTestImage
]

In [71]:
test_dataset = tf.data.Dataset.from_tensor_slices((camoTestImagePaths, camoTestMaskPaths))
test_dataset = test_dataset.map(preprocess_image)
test_dataset = test_dataset.batch(8)
test_dataset = test_dataset.prefetch(1)

In [72]:
loadedsegmentationModel.evaluate(test_dataset)

42/42 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - e_similarity: 0.8791 - loss: 0.6607 - mae: 0.0523 - s_score: 0.8519 - weighted_f_score: 0.8797


[0.6504245400428772,
 0.051629871129989624,
 0.8493016362190247,
 0.8765861988067627,
 0.882453441619873]

mae:                        0.0516

s_score (SScore):           0.8493

weighted_f_score:           0.8766

e_similarity:               0.8825

In [73]:
display_multiple_batches(loadedsegmentationModel, test_dataset, num_batches=6, num_images_per_batch=4)

Output hidden; open in https://colab.research.google.com to view.